# Lesson 2: Tool Calling

In [1]:
# if you want to see the timer of your ouput running
!pip install ipywidgets widgetsnbextension

In [2]:
!pip install groq

In [3]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

Groq_api_key = os.environ.get("GROQ_API_KEY")  # ← get the value, don't hardcode the name

client = Groq(
    api_key=Groq_api_key
)

In [4]:
import nest_asyncio
nest_asyncio.apply()

# 1. Define a Simple Tool

In [5]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y

def mystery(x: int, y: int) -> int: 
    """Mystery function that operates on top of two numbers."""
    return (x + y) * (x + y)

add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [6]:
from llama_index.llms.groq import Groq

llm = Groq(
    model="llama-3.3-70b-versatile"
)

response = llm.predict_and_call(
    [add_tool, mystery_tool], 
    "Tell me the output of the mystery function on 2 and 9", 
    verbose=True
)

print(str(response))

=== Calling Function ===
Calling function: mystery with args: {"x": 2, "y": 9}
=== Function Output ===
121
121


# 2. Define an Auto-Retrieval Tool
Load Data
To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

In [7]:
from llama_index.core import SimpleDirectoryReader
# load documents
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

In [8]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [11]:
print(nodes[0].get_content)

<bound method TextNode.get_content of TextNode(id_='77d4ce12-63d8-4b14-8f6b-25e1eca25a3f', embedding=None, metadata={'page_label': '1', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-05-23', 'last_modified_date': '2026-05-23'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='55acb57a-6cfa-45b8-b59b-2057680e9cbd', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-05-23', 'last_modified_date': '2026-05-23'}, hash='5689967a42e452fb8a65176165bb9874829f52a450617c086fbebee44edff4

In [12]:
print(nodes[0].get_content(metadata_mode="all"))

page_label: 1
file_name: metagpt.pdf
file_path: metagpt.pdf
file_type: application/pdf
file_size: 16911937
creation_date: 2026-05-23
last_modified_date: 2026-05-23

Preprint
METAGPT: M ETA PROGRAMMING FOR A
MULTI-AGENT COLLABORATIVE FRAMEWORK
Sirui Hong1∗, Mingchen Zhuge2∗, Jonathan Chen1, Xiawu Zheng3, Yuheng Cheng4,
Ceyao Zhang4, Jinlin Wang1, Zili Wang, Steven Ka Shing Yau5, Zijuan Lin4,
Liyang Zhou6, Chenyu Ran1, Lingfeng Xiao1,7, Chenglin Wu1†, J¨urgen Schmidhuber2,8
1DeepWisdom, 2AI Initiative, King Abdullah University of Science and Technology,
3Xiamen University, 4The Chinese University of Hong Kong, Shenzhen,
5Nanjing University, 6University of Pennsylvania,
7University of California, Berkeley, 8The Swiss AI Lab IDSIA/USI/SUPSI
ABSTRACT
Remarkable progress has been made on automated problem solving through so-
cieties of agents based on large language models (LLMs). Existing LLM-based
multi-agent systems can already solve simple dialogue tasks. Solutions to more
complex tasks,

In [14]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Groq(
    model="llama-3.3-70b-versatile"
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

Settings.chunk_size = 256
Settings.chunk_overlap = 20

2026-05-23 07:16:04,017 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-05-23 07:16:16,031 - INFO - Loaded 1 prompt with these keys: ['query']


In [15]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

In [17]:
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters = MetadataFilters.from_dicts(
        [
            {"key" : "page_label" , "value": "2"}
        ]
    )
)
response = query_engine.query(
    "What are some high-level results of MetaGPT?", 
)

2026-05-23 07:21:13,991 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [18]:
print(str(response))

MetaGPT achieves a new state-of-the-art with 85.9% and 87.7% in Pass@1 in code generation benchmarks. It also stands out in handling higher levels of software complexity and offering extensive functionality, with a 100% task completion rate, demonstrating the robustness and efficiency of its design.


In [21]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-05-23', 'last_modified_date': '2026-05-23'}


# Define the Auto-Retrieval Tool

In [23]:
from typing import List
from llama_index.core.vector_stores import FilterCondition

def vector_query(
    query: str, 
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    """

    metadata_dicts = [
        {"key" : "page_label" , "value" : p} for p in page_numbers
    ]

    query_engine = vector_index.as_query_engine(
        similarity_top_k = 2,
        filters = MetadataFilters.from_dicts(
            metadata_dicts,
            condition = FilterCondition.OR
        )    
    )

    response = query_engine.query(query)
    return response

vector_query_tool = FunctionTool.from_defaults(
    name = "vector_tool",
    fn = vector_query 
)
    

In [25]:
from llama_index.llms.groq import Groq

llm = Groq(
    model="llama-3.3-70b-versatile"
)

response = llm.predict_and_call(
    [vector_query_tool], 
    "What are the high-level results of MetaGPT as described on page 2?", 
    verbose=True
)

print(str(response))

2026-05-23 07:58:45,424 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: vector_tool with args: {"page_numbers": ["2"], "query": "MetaGPT high-level results"}


2026-05-23 07:58:47,727 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT achieves a new state-of-the-art with 85.9% and 87.7% in Pass@1 in code generation benchmarks. It also stands out in handling higher levels of software complexity and offering extensive functionality, with a 100% task completion rate, demonstrating the robustness and efficiency of its design.
MetaGPT achieves a new state-of-the-art with 85.9% and 87.7% in Pass@1 in code generation benchmarks. It also stands out in handling higher levels of software complexity and offering extensive functionality, with a 100% task completion rate, demonstrating the robustness and efficiency of its design.


In [26]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-05-23', 'last_modified_date': '2026-05-23'}


# Let's add some other tools!

In [28]:
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=(
        "Useful if you want to get a summary of MetaGPT"
    ),
)

In [29]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What are the MetaGPT comparisons with ChatDev described on page 8?", 
    verbose=True
)

2026-05-23 08:03:54,280 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: vector_tool with args: {"page_numbers": ["8"], "query": "MetaGPT comparisons with ChatDev"}


2026-05-23 08:03:55,894 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT outperforms ChatDev on the SoftwareDev dataset in nearly all metrics. For example, MetaGPT achieves a higher executability score of 3.75, compared to ChatDev's 2.25. Additionally, MetaGPT takes less time, with 503 seconds, compared to ChatDev's 762 seconds. MetaGPT also significantly outperforms ChatDev in terms of code statistics and the cost of human revision. However, MetaGPT requires more tokens, with 24,613 or 31,255, compared to ChatDev's 19,292. Nevertheless, MetaGPT needs only 126.5/124.3 tokens to generate one line of code, whereas ChatDev uses 248.9 tokens.


In [30]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '8', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-05-23', 'last_modified_date': '2026-05-23'}


In [31]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What is a summary of the paper?", 
    verbose=True
)

2026-05-23 08:04:18,775 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: summary_tool with args: {"input": "paper"}


2026-05-23 08:04:20,613 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-23 08:04:20,944 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-23 08:04:20,947 - INFO - Retrying request to /chat/completions in 19.000000 seconds
2026-05-23 08:04:21,432 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-05-23 08:04:21,580 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-23 08:04:21,598 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-23 08:04:21,707 - INFO - Retrying request to /chat/completions in 6.000000 seconds
2026-05-23 08:04:21,941 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-23 08:04:22,595 - INFO - HTTP Request: POST https://api.groq.com/openai

=== Function Output ===
A document or written work, such as a research paper, that presents information, results, or discussions on a particular topic, including those related to artificial intelligence, machine learning, and large language models, like the MetaGPT framework.
